In [1]:
import sys 

sys.path.append("..")

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import torch.backends.cudnn as cudnn
import random
from mamba_ssm import Mamba

from thop import clever_format
from ptflops import get_model_complexity_info

import os
os.chdir("/workspace/dehazing")

In [2]:
def set_seed(seed):
    """Sets the seed for reproducibility across random, numpy, and torch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        cudnn.deterministic = True
        cudnn.benchmark = False

## Stage 1 Code

In [3]:
def get_pad_layer(pad_type):
    if(pad_type in ['refl','reflect']):
        PadLayer = nn.ReflectionPad2d
    elif(pad_type in ['repl','replicate']):
        PadLayer = nn.ReplicationPad2da
    elif(pad_type=='zero'):
        PadLayer = nn.ZeroPad2d
    else:
        print(f'Pad type [{pad_type}] not recognized')
    return PadLayer


class AntiAlias_Downsample(nn.Module):
    def __init__(self, channels, pad_type = 'reflect', filt_size = 3, 
                        stride = 2, pad_off = 0):
        super(AntiAlias_Downsample, self).__init__()
        self.filt_size = filt_size
        self.pad_off = pad_off
        self.pad_type = pad_type

        # Asymmetric padding (round up at the top and round down at the bottom)
        # Perfect when kernel size is 2 
        self.pad_sizes = [int(1. * (filt_size - 1) / 2), int(np.ceil(1. * (filt_size - 1) / 2)),
                          int(1. * (filt_size - 1) / 2), int(np.ceil(1. * (filt_size - 1) / 2))]
        self.pad_sizes = [pad_size + pad_off for pad_size in self.pad_sizes]
        self.stride = stride 
        self.off = int((self.stride - 1) / 2.)
        self.channels = channels 

        # Define the binomial filter weights
        if(self.filt_size==1):
            a = np.array([1.,])
        elif(self.filt_size==2):
            a = np.array([1., 1.])
        elif(self.filt_size==3):
            a = np.array([1., 2., 1.])
        elif(self.filt_size==4):    
            a = np.array([1., 3., 3., 1.])
        elif(self.filt_size==5):    
            a = np.array([1., 4., 6., 4., 1.])
        elif(self.filt_size==6):    
            a = np.array([1., 5., 10., 10., 5., 1.])
        elif(self.filt_size==7):    
            a = np.array([1., 6., 15., 20., 15., 6., 1.])
            
        # Create a 2D filter by taking the outer product of the 1D filter
        filt = torch.tensor(a[:, None] * a[None, :], dtype = torch.float32)
        filt = filt / torch.sum(filt) # Normalize

        # Reshape to (out_channels, in_channels/groups, kH, kW) for 
        # depthwise convolution
        filt = filt.view(1, 1, filt_size, filt_size)
        filt = filt.repeat(channels, 1, 1, 1)

        # Register as a buffer so PyTorch knows these are NOT trainable parameters
        self.register_buffer('filt', filt)
        self.pad = get_pad_layer(pad_type)(self.pad_sizes)

    def forward(self, inp):
        if (self.filt_size == 1):
            if (self.pad_off == 0):
                return inp[:, :, ::self.stride, ::self.stride] 
            else:
                return self.pad(inp)[:, :, ::self.stride, ::self.stride] 

        else:
            return F.conv2d(self.pad(inp), self.filt, stride = self.stride, groups = inp.shape[1])


class VariantB_AntiAliasedDownsample(nn.Module):
    """
    Anti-Aliased Downsampling (BlurPool) based on Richard Zhang's paper.
    Preserves shift-invariance and prevents high-frequency aliasing.
    """
    def __init__(self, dim_in, dim_out):
        super().__init__()
        # 1. Feature Mixing (Stride 1 preserves all spatial information)
        self.conv = nn.Conv2d(dim_in, dim_out, kernel_size=3, stride=1, padding=1)
        # 2. Anti-aliased spatial reduction (Low-pass filter + subsampling)
        # NOTE: Make sure your AntiAlias_Downsample class is defined in the script!
        self.aa_down = AntiAlias_Downsample(channels=dim_out, filt_size=3, stride=2)

    def forward(self, x):
        return self.aa_down(self.conv(x))

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Stage1_DCP_Prior(nn.Module):
    """
    Refactored for Structural Sharpness:
    - Replaces Dilation with Depthwise Separable Bottleneck (Preserves local edges).
    - Swaps Bilinear Upsampling for PixelShuffle (Prevents interpolation blur).
    - Uses a Gated Residual connection for the Transmission map.
    """
    def __init__(self, in_channels=4, base_dim=32):
        super().__init__()
        self.variant = 'S_Sharp' # 'S' for Structural Sharpness
        
        # 1. Initial Encoder (RGB + DCP)
        self.init_conv = nn.Conv2d(in_channels, base_dim, kernel_size=3, padding=1)
        self.enc1 = nn.Conv2d(base_dim, base_dim, kernel_size=3, padding=1)
        self.enc2 = nn.Conv2d(base_dim * 2, base_dim * 2, kernel_size=3, padding=1)

        # 2. Downsampling (Anti-Aliased)
        self.down1 = VariantB_AntiAliasedDownsample(base_dim, base_dim * 2)
        self.down2 = VariantB_AntiAliasedDownsample(base_dim * 2, base_dim * 4)

        # 3. Dilated Bottleneck (1 -> 2 -> 1 Dilation Pattern)
        # This expands the receptive field to capture "haze context" 
        # while maintaining local edge precision.
        self.bottleneck = nn.Sequential(
            # Dilation 1: Local context
            nn.Conv2d(base_dim * 4, base_dim * 4, kernel_size=3, padding=1, dilation=1),
            nn.BatchNorm2d(base_dim * 4),
            nn.LeakyReLU(0.2, inplace=True),
            
            # Dilation 2: Mid-range context (Captures haze gradients)
            nn.Conv2d(base_dim * 4, base_dim * 4, kernel_size=3, padding=2, dilation=2),
            nn.BatchNorm2d(base_dim * 4),
            nn.LeakyReLU(0.2, inplace=True),
            
            # Dilation 1: Refine back to local structural details
            nn.Conv2d(base_dim * 4, base_dim * 4, kernel_size=3, padding=1, dilation=1),
            nn.BatchNorm2d(base_dim * 4),
            nn.LeakyReLU(0.2, inplace=True)
        )        
        
        # 4. Sharp Decoder (Using PixelShuffle to avoid interpolation blur)
        # Note: PixelShuffle(upscale_factor=2) reduces channels by 4x
        self.up1_ps = nn.Sequential(
            nn.Conv2d(base_dim * 4, base_dim * 8, kernel_size=1),
            nn.PixelShuffle(2)
        ) # Result: base_dim * 2 channels
        
        self.dec1_fusion = nn.Conv2d(base_dim * 4, base_dim * 2, kernel_size=1) 
        self.dec1 = nn.Conv2d(base_dim * 2, base_dim * 2, kernel_size=3, padding=1)
        
        self.up2_ps = nn.Sequential(
            nn.Conv2d(base_dim * 2, base_dim * 4, kernel_size=1),
            nn.PixelShuffle(2)
        ) # Result: base_dim channels
        
        self.dec2_fusion = nn.Conv2d(base_dim * 2, base_dim, kernel_size=1)
        self.dec2 = nn.Conv2d(base_dim, base_dim, kernel_size=3, padding=1)
        
        # 5. Output Heads
        self.t_head_residual = nn.Sequential(
            nn.Conv2d(base_dim, 1, kernel_size=3, padding=1), 
            nn.Tanh() 
        )

        self.A_head_spatial = nn.Sequential(
            nn.Conv2d(base_dim, 3, kernel_size=3, padding=1), 
            nn.Sigmoid() 
        )

        # self.A_head_global = nn.Sequential(
        #     nn.AdaptiveAvgPool2d(1),
        #     nn.Flatten(),
        #     nn.Linear(base_dim, base_dim // 2),
        #     nn.ReLU(inplace=True),
        #     nn.Linear(base_dim // 2, 3),
        #     nn.Sigmoid()
        # )

    def forward(self, hazy_img, t_dcp):
        x = torch.cat([hazy_img, t_dcp], dim=1)

        # Encoder
        x = F.relu(self.init_conv(x))
        e1 = F.relu(self.enc1(x))
        d1 = self.down1(e1)
        e2 = F.relu(self.enc2(d1))
        d2 = self.down2(e2)

        # Bottleneck (Preserves structural edges)
        b = self.bottleneck(d2)
        
        # Decoder 1: PixelShuffle + Skip Connection
        u1 = torch.cat([self.up1_ps(b), e2], dim=1)
        u1 = F.relu(self.dec1_fusion(u1))
        u1 = F.relu(self.dec1(u1))
        
        # Decoder 2: PixelShuffle + Skip Connection
        u2 = torch.cat([self.up2_ps(u1), e1], dim=1)
        u2 = F.relu(self.dec2_fusion(u2))
        u2 = F.relu(self.dec2(u2))

        # --- FINAL OUTPUTS ---
        t_residual = self.t_head_residual(u2)
        
        # Final Transmission = DCP Prior + Learned Residual
        t_final = torch.clamp(t_dcp + t_residual, 0.0, 1.0)
        A_spatial = self.A_head_spatial(u2)

        return t_final, A_spatial

## Stage 2 Code

**PhysConvNeXtBlock**

In [5]:
class PhysConvNeXtBlock(nn.Module):
    """
    ConvNeXt V2 Block: Best for local textures and edges.
    Includes Adaptive Layer Norm (AdaLN) for Time Embedding injection.
    """
    def __init__(self, dim, mult=2):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm = nn.LayerNorm(dim, eps=1e-6)
        self.pwconv1 = nn.Linear(dim, 4 * dim) 
        self.act = nn.GELU()
        self.pwconv2 = nn.Linear(4 * dim, dim)
        self.gamma = nn.Parameter(1e-6 * torch.ones((dim)), requires_grad=True)

    def forward(self, x, t_emb=None):
        inp = x
        x = self.dwconv(x)
        x = x.permute(0, 2, 3, 1)
        
        if t_emb is not None:
            x = self.norm(x)
            scale, shift = t_emb.chunk(2, dim=1)
            x = x * (1 + scale.unsqueeze(1).unsqueeze(1)) + shift.unsqueeze(1).unsqueeze(1)
        else:
            x = self.norm(x)
            
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        x = self.gamma * x
        x = x.permute(0, 3, 1, 2)
        return inp + x


**PhysBiMambaBlock**

In [6]:
class LocalFeatureExtractor(nn.Module):
    """ 
    Adaptive Parallel Branch.
    Uses Inverted Bottleneck (Expand -> Depthwise -> Project).
    Automatically calculates padding to keep spatial dimensions constant.
    """
    def __init__(self, dim, kernel_size=3, expansion_factor=2, dilation=2):
        super().__init__()
        
        hidden_dim = int(dim * expansion_factor)
        
        # Dynamic Padding Calculation:
        # P = (dilation * (kernel_size - 1)) / 2
        # This ensures the output size equals the input size.
        padding = (dilation * (kernel_size - 1)) // 2
        
        self.net = nn.Sequential(
            # 1. Pointwise Expansion
            nn.Conv2d(dim, hidden_dim, kernel_size=1),
            nn.GELU(),
            
            # 2. Adaptive Depthwise Conv
            nn.Conv2d(hidden_dim, hidden_dim, 
                      kernel_size=kernel_size, 
                      padding=padding, 
                      dilation=dilation,
                      groups=hidden_dim), # Depthwise
            nn.GELU(),
            
            # 3. Pointwise Projection
            nn.Conv2d(hidden_dim, dim, kernel_size=1)
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
class PhysBiMambaBlock(nn.Module):
    """
    Bidirectional Mamba Block (BiMamba)
    Scans the image Forward AND Backward so the top-left pixel
    can 'see' the bottom-right pixel.
    """
    def __init__(self, dim, dropout = 0.05):
        super().__init__()
        self.norm = nn.LayerNorm(dim)

        # --- Horizontal Mamba -----
        self.mamba_h_fwd = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)
        self.mamba_h_bwd = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)

        # --- Vertical Mamba ---
        self.mamba_v_fwd = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)
        self.mamba_v_bwd = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)
        
        # Fuses Fwd+Bwd direction
        self.fusion_linear = nn.Linear(dim * 4, dim)

        self.local_conv = LocalFeatureExtractor(dim, 
                                                kernel_size=3, 
                                                dilation=1)
        
        # Optional: A Gate to let the network choose emphasis
        self.mixer = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.Sigmoid()
        )

        self.out_proj = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, t_emb=None):
        B, C, H, W = x.shape
        residual = x
        
        x_flat = x.flatten(2).transpose(1, 2)
        x_norm = self.norm(x_flat)

        if t_emb is not None:
            scale, shift = t_emb.chunk(2, dim=1)
            x_norm = x_norm * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)

        # ---------------------------------------------------------
        # 2. HORIZONTAL SCANS (Raster Order)
        # ---------------------------------------------------------
        # Forward ->
        out_h_fwd = self.mamba_h_fwd(x_norm)
        
        # Backward <-
        x_flip = torch.flip(x_norm, dims=[1])
        out_h_bwd = self.mamba_h_bwd(x_flip)
        out_h_bwd = torch.flip(out_h_bwd, dims=[1]) # Flip back

        # ---------------------------------------------------------
        # 3. VERTICAL SCANS (Column-Major Order)
        # ---------------------------------------------------------
        # Reshape to Image -> Transpose (Swap H and W) -> Flatten
        # Result: (B, W*H, C). Now 'neighbors' in seq are vertical neighbors.
        x_v_img = x_norm.view(B, H, W, C).permute(0, 2, 1, 3) 
        x_v_flat = x_v_img.flatten(1, 2)
        
        # Down v
        out_v_fwd = self.mamba_v_fwd(x_v_flat)
        
        # Up ^
        x_v_flip = torch.flip(x_v_flat, dims=[1])
        out_v_bwd = self.mamba_v_bwd(x_v_flip)
        out_v_bwd = torch.flip(out_v_bwd, dims=[1])
        
        # Un-Transpose Vertical Outputs back to Horizontal Order
        # (B, W*H, C) -> (B, W, H, C) -> (B, H, W, C) -> (B, L, C)
        out_v_fwd = out_v_fwd.view(B, W, H, C).permute(0, 2, 1, 3).flatten(1, 2)
        out_v_bwd = out_v_bwd.view(B, W, H, C).permute(0, 2, 1, 3).flatten(1, 2)
        
        ## ---------------------------------------------------------
        # 4. Global Fusion
        # ---------------------------------------------------------
        # Combine all 4 views of the image
        global_feat = self.fusion_linear(
            torch.cat([out_h_fwd, out_h_bwd, out_v_fwd, out_v_bwd], dim=-1)
        )

        # ---------------------------------------------------------
        # 5. Local Branch (Conv)
        # ---------------------------------------------------------
        # Reshape for Conv2d
        x_img_norm = x_norm.transpose(1, 2).view(B, C, H, W)
        local_feat = self.local_conv(x_img_norm)
        local_feat = local_feat.flatten(2).transpose(1, 2)

        
        # ---------------------------------------------------------
        # 6. Gated Output
        # ---------------------------------------------------------
        combined = torch.cat([global_feat, local_feat], dim=-1)
        z = self.mixer(combined)
        
        fused = global_feat * z + local_feat * (1 - z)
        
        x_out = self.out_proj(fused)
        
        # Reshape to (B, C, H, W) for residual add
        x_out = x_out.transpose(1, 2).view(B, C, H, W)
        x_out = self.dropout(x_out)
        
        return residual + x_out


In [ ]:
class CAGatedFusion(nn.Module):
    """
    Channel Attention Gating (Version 2).
    Decides 'what features' (texture vs fog) to fuse using Global Context.
    """
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),          # Squeeze (Global Context)
            nn.Conv2d(dim * 2, dim // 2, 1),  # Compress
            nn.ReLU(inplace=True),
            nn.Conv2d(dim // 2, dim * 2, 1),  # Excite
            nn.Sigmoid()                      # Weight
        )
        self.conv = nn.Conv2d(dim, dim, 1)

    def forward(self, dec_feat, enc_feat):
        combined = torch.cat([dec_feat, enc_feat], dim=1)
        weights = self.attn(combined)
        w_dec, w_enc = weights.chunk(2, dim=1)
        # Channel-wise weighted fusion
        fused = (dec_feat * w_dec) + (enc_feat * w_enc)
        return self.conv(fused)


In [ ]:
class PixelShuffleUpsample(nn.Module):
    """
    SOTA Trick: Replaces ConvTranspose2d to eliminate checkerboard artifacts.
    """
    def __init__(self, dim_in, dim_out):
        super().__init__()
        # We need to project to (dim_out * 4) so PixelShuffle(2) results in dim_out
        self.conv = nn.Conv2d(dim_in, dim_out * 4, 3, 1, 1)
        self.pixel_shuffle = nn.PixelShuffle(2) # Scale x2
        
    def forward(self, x):
        return self.pixel_shuffle(self.conv(x))


## Flow Matching UNet

In [ ]:
class Stage2_FlowMatching_UNet(nn.Module):
    def __init__(self, 
                 in_channels = 7,     # x_t (3) + hazy_img (3) + frozen_t_map (1)
                 out_channels = 3,    # predicted vector field v (3)
                 base_dim = 64, 
                 dim_mults = [1, 2, 4, 8],
                 enc_blocks = [2, 2, 4], 
                 dec_blocks = [4, 2, 2],
                 num_mid_blocks = 4,
                 physics_guided=True):
        super().__init__()

        self.dims = [base_dim * m for m in dim_mults]
        self.physics_guided = physics_guided

        # --- Time & Physics Embedding (Global Conditioning) --- 
        time_dim = base_dim * 4
        self.time_mlp = nn.Sequential(
            nn.Linear(base_dim, time_dim), 
            nn.SiLU(), 
            nn.Linear(time_dim, time_dim)
        )

        # Fuses the Flow Timestep with the Frozen Atmospheric Light (A)
        self.phys_gate = nn.Sequential(
            nn.Linear(time_dim + 3, time_dim), 
            nn.SiLU(), 
            nn.Linear(time_dim, time_dim)
        )

        self.down_time_projs = nn.ModuleList()
        self.up_time_projs = nn.ModuleList(

        for i in range(len(self.dims) - 1):
            dim_in, dim_out = self.dims[i], self.dims[i + 1]
            self.down_time_projs.append(nn.Linear(time_dim, dim_in * 2))

            blocks = nn.ModuleList([PhysConvNeXtBlock(dim_in)])
            num_mamba = enc_blocks[i] if i < len(enc_blocks) else 1
            for _ in range(num_mamba):
                blocks.append(PhysBiMambaBlock(dim_in))

            self.downs.append(blocks)
            self.downsamples.append(nn.Conv2d(dim_in, dim_out, 4, 2, 1))

        # --- BOTTLENECK ---
        mid_dim = self.dims[-1]
        self.mid_time_proj = nn.Linear(time_dim, mid_dim * 2)
        self.mid_blocks = nn.ModuleList()
        for _ in range(num_mid_blocks):
            self.mid_blocks.append(PhysBiMambaBlock(mid_dim))

        # --- DECODER ---
        self.ups = nn.ModuleList()
        self.up_samples = nn.ModuleList()
        self.gates = nn.ModuleList()

        for idx, i in enumerate(range(len(self.dims) - 2, -1, -1)):
            dim_in, dim_out = self.dims[i+1], self.dims[i]
            self.up_time_projs.append(nn.Linear(time_dim, dim_out * 2))

            self.up_samples.append(PixelShuffleUpsample(dim_in, dim_out))
            self.gates.append(CAGatedFusion(dim_out))

            layers = nn.ModuleList()
            num_mamba = dec_blocks[idx] if idx < len(dec_blocks) else 1
            for _ in range(num_mamba):
                if i > 0: layers.append(PhysBiMambaBlock(dim_out))
                else: layers.append(PhysConvNeXtBlock(dim_out))

            layers.append(PhysConvNeXtBlock(dim_out)) 
            self.ups.append(layers)

        # Output Projection for the vector field
        self.final_conv = nn.Conv2d(self.dims[0], out_channels, 1)

    def get_sinusoidal_emb(self, t, device):
        half_dim = self.dims[0] // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = t[:, None] * emb[None, :]
        emb = torch.cat((emb.sin(), emb.cos()), dim=-1)
        return emb

    def forward(self, x_t, timestep, hazy_img, t_map, A):
        """
        Args:
            x_t: Noisy image at current timestep (B, 3, H, W)
            timestep: Flow matching timestep (B,)
            hazy_img: Original hazy input (B, 3, H, W)
            t_map: Stage 1 Transmission Map (B, 1, H, W)
            A: Stage 1 Atmospheric Light (B, 3)
        """
        # 1. Global Time & Physics Embedding 
        t_emb_raw = self.get_sinusoidal_emb(timestep, x_t.device) 
        t_vec = self.time_mlp(t_emb_raw)

        # Inject global atmospheric light 'A' into the flow timeline
        A = A.view(-1, 3)
        phys_cond = torch.cat([t_vec, A], dim = -1)
        t_vec = self.phys_g